# Load the data

In [40]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


file_path = "./Dataset/training/la_mmh_training_lms_item_activity_by_session_fact.csv"
df_lms = pd.read_csv(file_path)
print(df_lms.shape)

file_path_2 = "./Dataset/training/la_mmh_training_video_activity_by_view_fact.csv"
df_activity = pd.read_csv(file_path_2)
print(df_activity.shape)

file_path_3 = "./Dataset/training/la_mmh_training_person_dim.csv"
df_deregistrered = pd.read_csv(file_path_3)
print(df_deregistrered.shape)


file_path_4 = "./Dataset/training/la_mmh_training_person_module_instance_fact.csv"
df_module = pd.read_csv(file_path_4)
print(df_module.shape)

(7857424, 8)
(1145358, 7)
(14951, 3)
(100611, 4)


# Find the phantom

## LMS

In [30]:
print("Shape:", df_lms.shape)
# Find duplicated rows
duplicated_rows = df_lms[df_lms.duplicated(keep=False)]

print(duplicated_rows.shape)
# Save them
duplicated_rows.to_csv("duplicated_in_lms.csv", index=False)

# Clean the duplicates
df_lms_cleaned = df_lms.drop_duplicates()
print("Shape:", df_lms_cleaned.shape)

Shape: (7857424, 8)
(126634, 8)
Shape: (7794107, 8)


## Video activity

In [31]:
print("Shape:", df_activity.shape)
# Find duplicated rows
duplicated_rows = df_activity[df_activity.duplicated(keep=False)]

print(duplicated_rows.shape)

# Save them
duplicated_rows.to_csv("duplicated_in_videoactivity.csv", index=False)

# Clean the duplicates
df_activity_cleaned = df_activity.drop_duplicates()
print("Shape:", df_activity_cleaned.shape)

Shape: (1145358, 7)
(13938, 7)
Shape: (1138389, 7)


# Find the undead students

In [ ]:
df_deregistered_only = df_deregistrered[
    df_deregistrered['deregistration_date'].notna() & (df_deregistrered['deregistration_date'] != 'null')
    ]

print(f"Total number of deregristered person : {len(df_deregistered_only)}")

#df_deregistered_only['deregistration_date'] = pd.to_datetime(df_deregistered_only['deregistration_date'])
#df_activity['DATE'] = pd.to_datetime(df_activity['DATE'])

merged_df = pd.merge(df_activity, df_deregistered_only, on='P_ID', how='inner')

violations = merged_df[merged_df['DATE'] > merged_df['deregistration_date']]


# 1. Compter le nombre d'activités (lignes)
print(f"Nombre total d'activités trouvées après désinscription : {len(violations)}")

# 2. Compter le nombre d'étudiants UNIQUES concernés
unique_violators = violations['P_ID'].unique()
print(f"Nombre d'étudiants uniques actifs après désinscription : {len(unique_violators)}")

# 3. Sauvegarder les P_IDs uniques, chacun sur une ligne séparée (utile pour soumettre)
pd.DataFrame({'P_ID': unique_violators}).to_csv("students_active_after_deregistration_pids.csv", index=False)

# Sauvegarder le détail complet des anomalies si tu veux garder les lignes
violations.to_csv("activites_post_desinscription_anomalies.csv", index=False)


Total number of deregristered person : 196
Nombre total d'activités trouvées après désinscription : 961
Nombre d'étudiants uniques actifs après désinscription : 79


# Find the doppelganger

In [33]:
import pandas as pd

# 1. Créer une signature unique pour chaque action (ex: combinaison de VIDEO_ID et DATE ou TIMESTAMP)
# Tu peux ajuster les colonnes selon ce qui définit l'activité exacte
df_activity['activity_signature'] = (
    df_activity['VIDEO_ID'].astype(str) + '_' + 
    df_activity['DATE'].astype(str)
)

# 2. Regrouper par P_ID pour obtenir la liste complète (et triée) des activités de chaque étudiant
student_profiles = (
    df_activity.groupby('P_ID')['activity_signature']
    .apply(lambda x: tuple(sorted(list(x))))
    .reset_index(name='profile')
)

# 3. Trouver les profils identiques partagés par plusieurs P_ID
doppelgangers = student_profiles[
    student_profiles.duplicated(subset=['profile'], keep=False)
].sort_values('profile')

# --- Compter et afficher le nombre de doppelgängers ---
total_students = len(doppelgangers)
total_groups = doppelgangers['profile'].nunique()

print(f"🔍 Nombre total d'étudiants Doppelgängers : {total_students}")
print(f"👥 Nombre de paires/groupes identifiés : {total_groups}\n")

# 4. Afficher les P_IDs, chacun sur une ligne séparée (groupés par paire/groupe)
print("--- PAIRES DE DOPPELGÄNGERS ---")
for _, group in doppelgangers.groupby('profile'):
    for p_id in group['P_ID']:
        print(p_id)
    print("-" * 15)  # Séparateur visuel entre chaque paire

# Optionnel : Sauvegarder la liste directement dans un fichier CSV (avec chaque P_ID sur une ligne)
doppelgangers[['P_ID']].to_csv("doppelgangers_pids.csv", index=False)

🔍 Nombre total d'étudiants Doppelgängers : 98
👥 Nombre de paires/groupes identifiés : 45

--- PAIRES DE DOPPELGÄNGERS ---
P00907330
P00010770
---------------
P00012287
P00365560
---------------
P00922502
P00003382
---------------
P00948045
P00021322
---------------
P00336066
P00002959
---------------
P00019901
P00624727
---------------
P00219495
P00007309
---------------
P00017482
P00944765
---------------
P00408220
P00016841
---------------
P00000512
P00116388
---------------
P00004030
P00159448
---------------
P00010434
P00659653
---------------
P00013039
P00336081
---------------
P00955083
P00008029
---------------
P00959179
P00009037
---------------
P00006439
P00209491
---------------
P00901311
P00011616
---------------
P00647364
P00001387
---------------
P00011436
P00488189
---------------
P00003504
P00649204
---------------
P00243419
P00015570
---------------
P00019929
P00285442
---------------
P00000059
P00011201
---------------
P00959148
P00016398
---------------
P00014044
P005

# Find the SHAPE-SHIFTER

In [ ]:
import pandas as pd

# 1. Charger les tables nécessaires (adapte les chemins selon ton arborescence)
# df_module_registration contient : PMI_ID, P_ID, MODULE_INSTANCE_ID, MODULE_REGISTRATION_STATUS
print(df_module)

# 2. Si le programme est codé dans le MODULE_INSTANCE_ID ou via une table de référence :
# Par exemple, extrayons un préfixe ou identifiant de programme du MODULE_INSTANCE_ID 
# (À adapter selon la structure de tes IDs de modules, ex: les premiers caractères)
df_module['programme_proxy'] = df_module['MODULE_INSTANCE_ID'].astype(str).str.extract(r'([A-Za-z]+)')

# 3. Regrouper par étudiant et lister les programmes/modules uniques auxquels ils sont inscrits
student_programmes = df_module.groupby('P_ID')['programme_proxy'].unique().reset_index()

# 4. Filtrer les étudiants qui ont plus d'un programme distinct
shape_shifters = student_programmes[student_programmes['programme_proxy'].apply(len) > 1]

print(f"🔍 Nombre d'étudiants ayant changé de programme : {len(shape_shifters)}")
print(shape_shifters.head())

# 5. Sauvegarder la liste des P_ID (un par ligne) pour soumission
shape_shifters[['P_ID']].to_csv("shape_shifters_pids.csv", index=False)
print("Fichier 'shape_shifters_pids.csv' généré avec succès !")

            PMI_ID       P_ID MODULE_INSTANCE_ID MODULE_REGISTRATION_STATUS
0       PMI0153397  P00012615           I2301485                  Mandatory
1       PMI0062532  P00006447           I2301485                  Mandatory
2       PMI0090410  P00016549           I2302486                  Mandatory
3       PMI0074021  P00014949           I2301509                   Elective
4       PMI0082006  P00011861           I2302059                  Mandatory
...            ...        ...                ...                        ...
100606  PMI0119444  P00009944           I2300367                  Mandatory
100607  PMI0101257  P00008381           I2300367                  Mandatory
100608  PMI0128687  P00004674           I2301091                   Elective
100609  PMI0155492  P00017112           I2300367                  Mandatory
100610  PMI0109871  P00000753           I2300367                  Mandatory

[100611 rows x 4 columns]
🔍 Nombre d'étudiants ayant changé de programme : 0
Empty Data